# <font color='darkblue'>*Marina Napoleão - EBAC*</font>
## <font color='darkblue'>Projeto Cross Validation</font>

# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [2]:
base = pd.read_csv("Cientista de dados M35 - smoke_detection_iot.csv")

In [3]:
base.head(5)

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [4]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      62630 non-null  int64  
 1   UTC             62630 non-null  int64  
 2   Temperature[C]  62630 non-null  float64
 3   Humidity[%]     62630 non-null  float64
 4   TVOC[ppb]       62630 non-null  int64  
 5   eCO2[ppm]       62630 non-null  int64  
 6   Raw H2          62630 non-null  int64  
 7   Raw Ethanol     62630 non-null  int64  
 8   Pressure[hPa]   62630 non-null  float64
 9   PM1.0           62630 non-null  float64
 10  PM2.5           62630 non-null  float64
 11  NC0.5           62630 non-null  float64
 12  NC1.0           62630 non-null  float64
 13  NC2.5           62630 non-null  float64
 14  CNT             62630 non-null  int64  
 15  Fire Alarm      62630 non-null  int64  
dtypes: float64(8), int64(8)
memory usage: 7.6 MB


In [5]:
base.isnull().sum()

Unnamed: 0        0
UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw H2            0
Raw Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire Alarm        0
dtype: int64

In [6]:
base.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

In [7]:
base.dtypes

Unnamed: 0          int64
UTC                 int64
Temperature[C]    float64
Humidity[%]       float64
TVOC[ppb]           int64
eCO2[ppm]           int64
Raw H2              int64
Raw Ethanol         int64
Pressure[hPa]     float64
PM1.0             float64
PM2.5             float64
NC0.5             float64
NC1.0             float64
NC2.5             float64
CNT                 int64
Fire_Alarm          int64
dtype: object

Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.


* <font color='darkblue'>Escolhi usar dois modelos de previsão, a Regressão Logística(modelo linear) e o Random Forest(modelo não linear), assim posso comparar os resultados e verificar qual modelo melhor se adequa a base de dados.</font>

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [8]:
X = base.drop('Fire_Alarm', axis=1)
y = base['Fire_Alarm']

In [9]:
# Instanciando o modelo de regressão logística
modelo1 = LogisticRegression(max_iter=1000)

In [10]:
# Instanciando o modelo Random Forest
modelo2 = RandomForestClassifier(random_state = 42)

# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [11]:
folds = 5

In [12]:
crossvalidation = KFold(n_splits=folds, shuffle=True, random_state=5)

In [13]:
#Regressão logística
modelo1 = LogisticRegression(max_iter=1000)
modelo1_final = cross_val_score(modelo1, X, y, cv = folds)

In [14]:
#Random Forest
modelo2 = RandomForestClassifier(random_state = 42)
modelo2_final = cross_val_score(modelo2, X, y, cv = folds)

# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [15]:
pontuacoes = cross_val_score(modelo1, X, y, cv=crossvalidation)

print(f"Pontuações por fold Regressão logística: {pontuacoes}")

Pontuações por fold Regressão logística: [0.97620948 0.97573048 0.98826441 0.98626856 0.97684816]


In [16]:
pontuacoes = cross_val_score(modelo2, X, y, cv=crossvalidation)

print(f"Pontuações por fold Random Forest: {pontuacoes}")

Pontuações por fold Random Forest: [1.         1.         0.99992017 0.9997605  1.        ]


In [17]:
print('Regressão logística', modelo1_final.mean())

Regressão logística 0.9262334344563309


In [18]:
print('Random Forest:', modelo2_final.mean())

Random Forest: 0.9600191601468945


* <font color='darkblue'>Podemos observar que ambos os modelos apresentam bons resultados. A regressão logística obteve uma ótima média de 0.9262, enquanto o random florest alcançou uma excelente média de 0.9600. No entanto, é necessário manter atenção no random florest, pois isso pode indicar um possível overfitting.</font>